In [11]:
from transformers import AutoImageProcessor , AutoModelForImageClassification
import numpy as np
import pandas as pd
from sklearn.utils import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from torch.utils.data import Dataset,DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
import os
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

In [12]:
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")

In [13]:
def load_images_paths(root_dir):
    image_paths = []
    labels = []
    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

    for cls_name in classes:
        cls_folder = os.path.join(root_dir, cls_name)
        for fname in os.listdir(cls_folder):
            if fname.endswith('.jpg') or fname.endswith('.png'):
                image_paths.append(os.path.join(cls_folder, fname))
                labels.append(class_to_idx[cls_name])
    return image_paths, labels, class_to_idx

train_paths, train_labels, class_to_idx = load_images_paths("C:/train")
test_paths, test_labels, _ = load_images_paths("C:/test")

print(class_to_idx)
print(len(train_paths), len(test_paths))

{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
28709 7178


In [14]:
class EmoDataset():
    def __init__(self, image_paths, labels, processor):
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        labels = self.labels[idx]
        image = Image.open(image_path).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs['pixel_values'].squeeze(0)  
        
        return {
    "pixel_values": pixel_values,
    "labels": torch.tensor(labels, dtype=torch.long)
}

In [15]:
model = AutoModelForImageClassification.from_pretrained("microsoft/resnet-50", num_labels=7,ignore_mismatched_sizes=True)

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 2419.77it/s]
[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key                 | Status   |                                                                                            
--------------------+----------+--------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 2048]) vs model:torch.Size([7, 2048])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([7])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [16]:
device = torch.device('cuda')
model = model.to(device)
print(next(model.parameters()).device)

cuda:0


In [17]:
train_dataset = EmoDataset(train_paths, train_labels, processor)
test_dataset = EmoDataset(test_paths, test_labels, processor)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

train_labels = torch.tensor(train_labels, dtype=torch.long).to(device)
test_labels = torch.tensor(test_labels, dtype=torch.long).to(device)

y=train_labels.cpu().numpy()
class_weight=compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2,3,4,5,6]),
    y=y
)
class_weight=torch.tensor(class_weight,dtype=torch.float).to(device)

In [ ]:
# total_steps = len(train_loader) * 10
# warmup_steps = total_steps // 10

# optimizer=AdamW(model.parameters(),lr=1e-4)
# scheduler = get_linear_schedule_with_warmup(
#     optimizer,
#     num_warmup_steps=warmup_steps,
#     num_training_steps=total_steps
# )
# class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
# loss_fn=CrossEntropyLoss(weight=class_weight_tensor)

# for epoch in range(10):
#     print(f"Epoch {epoch+1}/10")
#     model.train()
#     total_loss = 0

#     for batch in train_loader:
#         pixel_values = batch["pixel_values"].to(device)
#         labels = batch["labels"].to(device)
#         optimizer.zero_grad()
#         forward_pass = model(pixel_values=pixel_values, labels=labels)
#         logits = forward_pass.logits
#         loss = loss_fn(logits, labels)
#         loss.backward()
#         optimizer.step()
#         scheduler.step()
#         total_loss += loss.item()
    
#     print(f"Training loss: {total_loss/len(train_loader)}")

In [19]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

processor = AutoImageProcessor.from_pretrained("face_model")
model = AutoModelForImageClassification.from_pretrained("face_model")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("Model loaded!")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 2897.33it/s]


Model loaded!


In [22]:
model.eval()
all_preds=[]
all_labels=[]

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(pixel_values=pixel_values)
        all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import classification_report, confusion_matrix
taget_names = list(class_to_idx.keys())
print(classification_report(all_labels, all_preds, target_names=taget_names))


              precision    recall  f1-score   support

       angry       0.49      0.51      0.50       958
     disgust       0.64      0.60      0.62       111
        fear       0.44      0.43      0.44      1024
       happy       0.89      0.78      0.83      1774
     neutral       0.56      0.55      0.56      1233
         sad       0.48      0.52      0.50      1247
    surprise       0.70      0.80      0.75       831

    accuracy                           0.61      7178
   macro avg       0.60      0.60      0.60      7178
weighted avg       0.62      0.61      0.61      7178

